# Phase B — Stage 4: merge → GGUF Q4_K_M → Hugging Face Hubทำกับ **ผู้ชนะจาก Stage 3 = `ft_case_a`** (typhoon2-qwen2.5-7b + LoRA)เหตุผลที่เลือก: judge เฉลี่ย 3.75/5 (= 91% ของ Gemini teacher), usable 98%, tone 4.37**Runtime ที่ต้องใช้:** GPU T4 (Runtime → Change runtime type → T4 GPU)**ผลลัพธ์:** ไฟล์ GGUF Q4_K_M (~4.4 GB) บน HF Hub + model card---### รันทีละหัวข้อ อย่ารันรวด — เช็คผลแต่ละขั้นก่อนไปต่อ| ขั้น | ใช้เวลาโดยประมาณ ||---|---|| 1-2 เตรียม + login | ~5 นาที || 3 โหลด adapter | ~5 นาที || 4 export GGUF (build llama.cpp ครั้งแรก) | **~30-45 นาที** || 5 sanity check หลัง quantize | ~5 นาที || 6-7 push + model card | ~15 นาที (ขึ้นกับเน็ต) |

## 1. ติดตั้ง + เตรียม path

In [ ]:
%%captureimport os!pip install --upgrade pip!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"!pip install --no-deps trl peft accelerate bitsandbytes xformers triton

In [ ]:
from google.colab import drivedrive.mount('/content/drive')DRIVE_ROOT   = '/content/drive/MyDrive/thai-paper-feed-phase-b'ADAPTER_DIR  = f'{DRIVE_ROOT}/adapters'WINNER       = f'{ADAPTER_DIR}/case_a_typhoon2_qwen25_7b'   # ผู้ชนะ Stage 3GGUF_DIR     = '/content/gguf_out'                          # เขียนลงดิสก์ Colab ไม่ใช่ Drive                                                            # (merged 16-bit ~15GB จะเต็ม Drive ฟรี)assert os.path.isdir(WINNER), f'ไม่เจอ adapter ที่ {WINNER}'print('adapter:', WINNER)print('ไฟล์ข้างใน:', sorted(os.listdir(WINNER)))

In [ ]:
# เช็คทรัพยากรก่อน — 7B ต้องการดิสก์เยอะ (merged fp16 ~15GB + GGUF ~4.4GB)!nvidia-smi --query-gpu=name,memory.total --format=csv,noheaderprint('--- ดิสก์ว่าง /content ---')!df -h /content | tail -1print('--- RAM ---')!free -g | head -2

## 2. Login Hugging Faceต้องใช้ token แบบ **write** — สร้างที่ https://huggingface.co/settings/tokens(อย่าพิมพ์ token ลงใน cell ตรงๆ ใช้ getpass เพื่อไม่ให้ค่าติดอยู่ใน notebook)

In [ ]:
from getpass import getpassfrom huggingface_hub import login, whoamilogin(getpass('วาง HF token (write) แล้วกด Enter: '))HF_USER = whoami()['name']REPO_ID = f'{HF_USER}/thai-paper-summarizer-7b'print('จะ push ไปที่:', REPO_ID)

## 3. โหลด adapterโหลดแบบ 4bit เหมือนตอนเทรน/eval — Unsloth อ่าน `adapter_config.json`แล้วดึง base model (`scb10x/typhoon2-qwen2.5-7b-instruct`) มาต่อให้เอง

In [ ]:
from unsloth import FastLanguageModelmodel, tokenizer = FastLanguageModel.from_pretrained(    model_name      = WINNER,    max_seq_length  = 3072,    dtype           = None,    load_in_4bit    = True,)print('โหลด adapter + base สำเร็จ')

## 4. Export เป็น GGUF Q4_K_M`save_pretrained_gguf` ทำ 3 อย่างให้ในคำสั่งเดียว:1. **merge** LoRA เข้า base → โมเดล 16-bit เต็มตัว2. clone + build **llama.cpp** (ครั้งแรกนานสุด ~20-30 นาที)3. **quantize** เป็น Q4_K_M> Q4_K_M = เล็กพอรันบน CPU ได้ คุณภาพดรอปน้อย — ถ้า sanity check (ขั้น 5) ออกมาแย่ ค่อยลอง `q5_k_m`

In [ ]:
model.save_pretrained_gguf(    GGUF_DIR,    tokenizer,    quantization_method = 'q4_k_m',)print('--- ไฟล์ที่ได้ ---')!ls -lh {GGUF_DIR}

## 5. Sanity check หลัง quantize (สำคัญ — อย่าข้าม)quantize = ตัดความละเอียดของ weight ทิ้ง คุณภาพ**อาจ**ดรอปเอา 3 ใบจาก test set มาให้โมเดล GGUF สรุป แล้วดูด้วยตาว่ายังเขียนไทยโทนเพื่อน + JSON ครบไหม(ไม่ใช่ eval เต็ม — eval เต็ม 60 ใบทำทีหลังได้ด้วยสคริปต์เดิมจาก Stage 3)

In [ ]:
import json, glob# หา binary ของ llama.cpp ที่ unsloth build ไว้CLI = Nonefor pat in ['/content/llama.cpp/llama-cli', '/content/llama.cpp/build/bin/llama-cli', '/content/llama.cpp/main']:    if glob.glob(pat):        CLI = glob.glob(pat)[0]        breakGGUF = sorted(glob.glob(f'{GGUF_DIR}/*.gguf'))[-1]print('cli :', CLI)print('gguf:', GGUF)

In [ ]:
def load_jsonl(path):    with open(path, encoding='utf-8') as f:        return [json.loads(l) for l in f if l.strip()]test_rows = load_jsonl(f'{DRIVE_ROOT}/data/test.jsonl')samples = test_rows[:3]print(f'จะลอง {len(samples)} ใบ:', [r['id'] for r in samples])

In [ ]:
import subprocessdef run_gguf(user_msg, max_tokens=1024):    prompt = tokenizer.apply_chat_template(        [{"role": "user", "content": user_msg}],        tokenize=False, add_generation_prompt=True,    )    out = subprocess.run(        [CLI, '-m', GGUF, '-p', prompt, '-n', str(max_tokens),         '--temp', '0', '-ngl', '99', '--no-display-prompt'],        capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=900,    )    return out.stdout.strip()for r in samples:    print('=' * 70)    print('id:', r['id'])    text = run_gguf(r['user'])    print(text[:600])    try:        obj = json.loads(text[text.index('{'):text.rindex('}') + 1])        missing = [k for k in ('title_th', 'summary_th', 'wow_point', 'tags') if not obj.get(k)]        print('>> JSON ok |', 'ครบทุก field' if not missing else f'ขาด {missing}')    except Exception as e:        print('>> JSON พัง:', e)

## 6. Push ขึ้น Hugging Face Hubpush **เฉพาะ GGUF** (~4.4GB) ไม่ push merged 16-bit (~15GB) เพราะ Stage 5 ใช้ GGUF อย่างเดียวถ้าอยากได้ 16-bit ไว้ต่อยอดค่อยกลับมา push ทีหลังด้วย `model.push_to_hub_merged(...)`

In [ ]:
from huggingface_hub import HfApiapi = HfApi()api.create_repo(REPO_ID, repo_type='model', exist_ok=True)api.upload_folder(    folder_path = GGUF_DIR,    repo_id     = REPO_ID,    repo_type   = 'model',    allow_patterns = ['*.gguf'],)print('เสร็จ ->', f'https://huggingface.co/{REPO_ID}')

## 7. Model card**นี่คือหน้าโชว์ portfolio** — คนที่เข้ามาเจอ repo จะอ่านอันนี้ก่อนตัวเลขทั้งหมดมาจาก `phase-b/eval_results/SCORECARD.md` ของจริง อย่าแต่งเพิ่ม

In [ ]:
CARD = '''---license: apache-2.0language: [th]base_model: scb10x/typhoon2-qwen2.5-7b-instructtags: [thai, summarization, arxiv, lora, gguf, distillation]---# Thai Paper Summarizer 7B (GGUF Q4_K_M)สรุป paper AI (title + abstract ภาษาอังกฤษ) ให้เป็น **การ์ดภาษาไทยโทนเพื่อนเล่าให้ฟัง**คืนค่าเป็น JSON: `title_th`, `summary_th`, `wow_point`, `tags`Fine-tune จาก `scb10x/typhoon2-qwen2.5-7b-instruct` ด้วย LoRA (r=16, alpha=16)โดย **distill จาก Gemini** (gemini-3.1-flash-lite) — 340 ตัวอย่างเทรน / 60 ตัวอย่างเทสต์## ผล Evaluation (test set 60 ใบ)ให้ **Claude Sonnet 5 เป็นกรรมการแบบ blind** (เห็นแค่ paper + การ์ด 1 ใบ ไม่รู้ว่าใครสรุป)ให้คะแนน 1-5 สามแกน — Gemini ถูกตัดสินด้วยเกณฑ์เดียวกัน| ระบบ | accuracy | tone | wow | เฉลี่ย ||---|:-:|:-:|:-:|:-:|| Gemini (teacher) | 3.95 | 4.68 | 3.73 | **4.12** || **โมเดลนี้** | 3.45 | 4.37 | 3.42 | **3.75** || base (ไม่ fine-tune) | 3.50 | 2.43 | 2.00 | **2.64** |**ได้ 91% ของ teacher** และดีกว่า base ชัดเจน### fine-tune ซื้ออะไรมา| | base | fine-tuned ||---|:-:|:-:|| tone (เพื่อนเล่า) | 2.43 | **4.37** || wow (จุดว้าว) | 2.00 | **3.42** || คงศัพท์อังกฤษ (en tokens) | 4 | **8** || JSON ใช้ได้จริง | 100% | 98% |น่าสนใจ: **accuracy แทบไม่ต่าง** (3.50 vs 3.45) — สิ่งที่ fine-tune แก้คือ *โทนและการเล่า*ไม่ใช่ความถูกต้อง base อ่าน paper รู้เรื่องอยู่แล้ว แค่เล่าแบบแปลตรงตัว/วิชาการจนไม่มีใครอยากอ่าน## วิธีใช้ (llama.cpp)```bashllama-cli -m thai-paper-summarizer-7b.Q4_K_M.gguf \  -p "Title: ...\n\nAbstract: ..." -n 1024 --temp 0```## ข้อจำกัด- เทรนจาก paper สาย **cs.CL / cs.AI** เท่านั้น — สาขาอื่นอาจเพี้ยน- dataset เล็ก (340 ตัวอย่าง) — เป็นงาน distill เชิงเรียนรู้ ไม่ใช่โมเดล production scale- ~2% ของ output ยัง JSON ไม่ครบ field ฝั่งที่เรียกใช้ควรมี fallback## TrainingUnsloth + QLoRA 4bit บน Colab T4 · eval_loss 0.987โปรเจกต์เต็ม: https://github.com/Tanapunn/thai-paper-feed'''with open('/content/README.md', 'w', encoding='utf-8') as f:    f.write(CARD)api.upload_file(    path_or_fileobj = '/content/README.md',    path_in_repo    = 'README.md',    repo_id         = REPO_ID,    repo_type       = 'model',)print('model card ขึ้นแล้ว ->', f'https://huggingface.co/{REPO_ID}')

## เสร็จ Stage 4- [ ] GGUF Q4_K_M อยู่บน HF Hub- [ ] model card มีตัวเลข eval จริง- [ ] sanity check ผ่าน (JSON ครบ + โทนไทยยังดี)**ต่อไป Stage 5:** เสียบเข้าเว็บ — GitHub Actions cron → โหลด GGUF จาก HF → llama.cpp สรุป → upsert Supabase(เก็บ Gemini ไว้เป็น fallback)